## Fase 0: split y limpieza de datos

**Qué se va a hacer:** se va a cargar el dataset original, crear la categoría de ingreso (`income_cat`) para poder hacer un split estratificado 80/20, separar features (`X`) de target (`y`) en train y test, y aplicar el filtro de outliers estructurales (topes artificiales de precio y edad vistos en el EDA) a ambos conjuntos — para dejar los datos listos antes de construir cualquier pipeline o probar modelos.


In [1]:
# Cargar el mismo dataset original, para construir el split y las funciones de limpieza
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/casas_california.csv")

In [2]:
# Crear categorias de ingreso (bins) para poder estratificar el split por nivel de ingreso
df["income_cat"] = pd.cut(df["median_income"], bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf], labels=[1, 2, 3, 4, 5])

In [3]:
# Split 80/20 estratificado por income_cat, para que train y test mantengan la misma proporcion de cada nivel de ingreso
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(df, test_size=0.2, random_state=42, stratify=df["income_cat"])

In [4]:
# Verificar que la estratificacion funciono: las proporciones de income_cat deben ser casi iguales en train y test
print(train_set["income_cat"].value_counts(normalize=True))
print(test_set["income_cat"].value_counts(normalize=True))

income_cat
3    0.350594
2    0.318859
4    0.176296
5    0.114462
1    0.039789
Name: proportion, dtype: float64
income_cat
3    0.350533
2    0.318798
4    0.176357
5    0.114341
1    0.039971
Name: proportion, dtype: float64


In [5]:
# Se elimina income_cat de ambos conjuntos de datos.
train_set = train_set.drop(["income_cat"], axis=1)
test_set = test_set.drop(["income_cat"], axis=1)

In [6]:
# Confirmar la proporcion 80/20 despues del split
print(train_set.shape)
print(test_set.shape)

(16512, 10)
(4128, 10)


In [7]:
# Separar features (X) y target (y) del train set
X_train = train_set.drop(["median_house_value"], axis=1)
y_train = train_set["median_house_value"]

In [8]:
# Confirmar dimensiones de X_train y y_train
print(X_train.shape)
print(y_train.shape)

(16512, 9)
(16512,)


In [9]:
# Separar features (X) y target (y) del test set
X_test = test_set.drop(["median_house_value"], axis=1)
y_test = test_set["median_house_value"]

In [10]:
# Confirmar dimensiones de X_train y y_train
print(X_test.shape)
print(y_test.shape)

(4128, 9)
(4128,)


In [11]:
# Agregar la carpeta padre del proyecto a sys.path, para poder importar desde src/ (carpeta hermana de notebooks/)
import sys
import os

sys.path.append(os.path.dirname(os.getcwd()))

In [12]:
# Importando la funcion para filtrar los outliers
from src.preprocesamiento import filtrar_outliers_estructurales

In [13]:
# Aplicar el filtro de outliers estructurales (regla fija) a train y test
X_train_filtrado, y_train_filtrado = filtrar_outliers_estructurales(X_train, y_train)
X_test_filtrado, y_test_filtrado = filtrar_outliers_estructurales(X_test, y_test)

In [14]:
print(X_train_filtrado.shape)
print(y_train_filtrado.shape)

print(X_test_filtrado.shape)
print(y_test_filtrado.shape)

(14839, 9)
(14839,)
(3733, 9)
(3733,)


In [15]:
# reseteando indices
X_train_filtrado = X_train_filtrado.reset_index(drop=True)
y_train_filtrado = y_train_filtrado.reset_index(drop=True)

X_test_filtrado = X_test_filtrado.reset_index(drop=True)
y_test_filtrado = y_test_filtrado.reset_index(drop=True)

## Fase 0: split y limpieza de datos (cierre)

**Conclusión:** quedan listos `X_train_filtrado`/`y_train_filtrado` (14,839 filas) y `X_test_filtrado`/`y_test_filtrado` (3,733 filas) — el split estratificado por nivel de ingreso, ya sin los outliers estructurales (tope de precio en 500,001 y tope de edad en 52), con los índices reiniciados. A partir de aquí, el test set queda guardado para usarse una sola vez, al final de esta ronda.


---
---

## Fase 1: construyendo el pipeline de preprocesamiento

**Qué se va a hacer:** antes de probar cualquier modelo, se va a armar el `ColumnTransformer` que preprocesa cada grupo de columnas según su naturaleza: las columnas con cola larga (sesgadas) se les aplica logaritmo antes de escalar, las numéricas sin sesgo solo se escalan, y la categórica (`ocean_proximity`) se codifica con one-hot. Esto evita tratar todas las columnas por igual cuando tienen distribuciones muy distintas.


In [ ]:
# Imports de sklearn para construir los pipelines, el ColumnTransformer, los modelos y las busquedas de hiperparametros
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import root_mean_squared_error

In [ ]:
# Sub-pipeline para columnas sesgadas: imputar nulos, aplicar log, escalar
pipeline_log = make_pipeline(SimpleImputer(strategy="median"),
                        FunctionTransformer(np.log, feature_names_out="one-to-one"),
                        StandardScaler())

# Numéricas sin sesgo
num_pipeline_simple = make_pipeline(SimpleImputer(strategy="median"),
                                    StandardScaler())

# Categoricas
cat_pipeline = make_pipeline(SimpleImputer(strategy="most_frequent"),
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False)) # sparse_output no es estrictamente necesario ColumnTransformer maneja la combinación de resultados dispersos y internamente


In [ ]:
# Nombres de columnas para cada grupo, usados por el ColumnTransformer
columnas_sesgadas = ["total_rooms", "total_bedrooms", "population", "households", "median_income"]
columnas_no_sesgadas = ['longitude', 'latitude', 'housing_median_age']
columna_categorica = ["ocean_proximity"]

In [19]:
# Contruir el ColumnTransformer
column_transformer = ColumnTransformer([
    ("num_log", pipeline_log, columnas_sesgadas),
    ("num_simple", num_pipeline_simple, columnas_no_sesgadas),
    ("cat", cat_pipeline, columna_categorica)
])

## Fase 1: construyendo el pipeline de preprocesamiento (cierre)

**Conclusión:** quedan listos los tres sub-pipelines (log, numérico simple, categórico) y el `ColumnTransformer` que los combina, aplicando cada transformación solo a las columnas que le corresponden. Con esto armado, se puede empezar a probar modelos.


---
---

## Probando con LinearRegression

**Qué se va a hacer:** entrenar y evaluar el primer modelo — una `LinearRegression` simple sobre las columnas transformadas por `column_transformer` — usando `cross_val_score` con `cv=10` para obtener un RMSE promedio confiable, y convertirlo a porcentaje sobre la mediana del precio para tener una referencia de línea base contra la cual comparar los modelos siguientes.


In [ ]:
# Modelo 1: ColumnTransformer + Regresion Lineal simple
pipeline_final = Pipeline([
    ("transformer", column_transformer),
    ("modelo", LinearRegression())
])

In [21]:
from sklearn.model_selection import cross_val_score

In [ ]:
# Evaluar Regresion Lineal con cross-validation (10 particiones), solo sobre train
evaluacion = cross_val_score(pipeline_final, X_train_filtrado, y_train_filtrado, cv=10, scoring="neg_root_mean_squared_error")

In [ ]:
# Convirtiendo el resultado a dolares
evaluacion_dolares = evaluacion * -1

In [ ]:
# Promedio y desviacion estandar del RMSE entre las 10 vueltas
rmse_promedio = evaluacion_dolares.mean()
rmse_std = evaluacion_dolares.std()

In [27]:
print(rmse_promedio)
print(rmse_std)

61243.68279926694
1992.3756159759243


In [ ]:
# Media y mediana del precio real, para usar como referencia de comparacion
media_precio = y_train_filtrado.mean()
mediana_precio = y_train_filtrado.median()

In [29]:
print(media_precio)
print(mediana_precio)

189466.25763191588
170800.0


In [ ]:
# RMSE como porcentaje de la mediana del precio - linea base del proyecto (35.9%)
porcentaje_precio = rmse_promedio / mediana_precio * 100
print(porcentaje_precio)

35.85695714242795


## Probando con LinearRegression — RESULTADO

**Resultado:** RMSE promedio de 35.86% de la mediana del precio — este es el resultado de línea base de todo el proyecto, sin regularización ni términos no lineales, que sirve como punto de comparación para todos los modelos que siguen.


---
---

### Probando con PolynomialFeatures

**Qué se va a hacer:** probar `PolynomialFeatures` (degree=2) combinado con `LinearRegression` — agrega columnas con productos e interacciones entre las variables originales, para ver si el modelo mejora cuando se le da la capacidad de capturar relaciones no lineales entre las columnas.


In [ ]:
# Modelo 2: ColumnTransformer + PolynomialFeatures (grado 2) + Regresion Lineal
pipeline_polinomal = Pipeline([
    ("transformer", column_transformer),
    ("polinomial", PolynomialFeatures(degree=2, include_bias=False)),
    ("regresor", LinearRegression())
])

In [34]:
evaluacion_poly = cross_val_score(pipeline_polinomal, X_train_filtrado, y_train_filtrado, cv=10, scoring="neg_root_mean_squared_error")

In [38]:
evaluacion_poly_dolares = evaluacion_poly * -1
print(evaluacion_poly_dolares)

[52965.96217304 50191.44450959 50209.63631038 55960.0432414
 50340.14836007 52918.87687334 54529.17306902 54294.74786918
 50747.77810962 53076.06296058]


In [39]:
rmse_promedio_poly = evaluacion_poly_dolares.mean()
rmse_std_poly = evaluacion_poly_dolares.std()
print(rmse_promedio_poly)
print(rmse_std_poly)


52523.387347621494
1958.0591122077615


In [40]:
media_precio_poly = y_train_filtrado.mean()
mediana_precio_poly = y_train_filtrado.median()
print(media_precio_poly)
print(mediana_precio_poly)

189466.25763191588
170800.0


In [ ]:
# Polynomial grado 2 mejora sobre la linea base (30.8% vs 35.9%)
porcentaje_promedio_poly = rmse_promedio_poly / mediana_precio_poly * 100
print(porcentaje_promedio_poly)

30.751397744509074


**Resultado:** RMSE promedio de 30.75% — mejora clara sobre la línea base de `LinearRegression` (35.86% → 30.75%). Agregar interacciones y términos cuadráticos entre las columnas (degree=2) le da al modelo la capacidad de capturar relaciones no lineales que una regresión lineal simple no puede representar.


---
---

### Probando Ridge

**Qué se va a hacer:** probar `Ridge` (alpha=1.0, sin `PolynomialFeatures` todavía) — agrega regularización L2 sobre `LinearRegression`, para ver si penalizar los coeficientes grandes mejora el resultado de la línea base por sí sola, antes de combinarlo con Polynomial.


In [ ]:
# Modelo 3: ColumnTransformer + Ridge sola (sin Polynomial), alpha por defecto
pipeline_ridge = Pipeline([
    ("transformer", column_transformer),
    ("regresor", Ridge(alpha=1.0))
])

In [45]:
evaluacion_ridge = cross_val_score(pipeline_ridge, X_train_filtrado, y_train_filtrado, cv=10, scoring="neg_root_mean_squared_error")

In [46]:
evaluacion_ridge_dolares = evaluacion_ridge * -1
print(evaluacion_ridge_dolares)

[62025.68380802 58704.32441889 58604.85565567 64243.25473762
 58808.9064789  61934.96813206 62486.57624004 64022.15650684
 60257.96970458 61347.2976112 ]


In [47]:
rmse_promedio_ridge = evaluacion_ridge_dolares.mean()
rmse_std_ridge = evaluacion_ridge_dolares.std()
print(rmse_promedio_ridge)
print(rmse_std_ridge)

61243.599329381235
1991.8279223799175


In [48]:
media_precio_ridge = y_train_filtrado.mean()
mediana_precio_ridge = y_train_filtrado.median()
print(media_precio_ridge)
print(mediana_precio_ridge)

189466.25763191588
170800.0


In [ ]:
# Ridge sola no mejora nada sobre la linea base: sin Polynomial, no hay pesos "rebeldes" que penalizar
porcentaje_promedio_ridge = rmse_promedio_ridge / mediana_precio_ridge * 100
print(porcentaje_promedio_ridge)

35.85690827247145


**Resultado:** RMSE promedio de 35.86% — prácticamente idéntico a `LinearRegression` (35.86%). Con `alpha=1.0` y sin `PolynomialFeatures`, Ridge no tiene "pesos rebeldes" que penalizar: el modelo sigue siendo lineal, así que la regularización sola no resuelve la falta de capacidad para capturar relaciones no lineales.


---
---

### Probando Ridge + PolynomialFeatures 

**Qué se va a hacer:** combinar `PolynomialFeatures` (degree=2) con `Ridge` (alpha=1.0) en el mismo pipeline, para ver si la regularización L2 mejora el resultado de Polynomial solo, controlando el riesgo de sobreajuste que trae el grado 2.


In [ ]:
# Modelo 4: ColumnTransformer + PolynomialFeatures (grado 2) + Ridge (veneno + antidoto)
pipeline_ridge_poly = Pipeline([
    ("transformer", column_transformer),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("ridge", Ridge(alpha=1.0))
])

In [52]:
evaluacion_ridge_poly = cross_val_score(pipeline_ridge_poly, X_train_filtrado, y_train_filtrado, cv=10, scoring="neg_root_mean_squared_error")

In [53]:
evaluacion_ridge_poly_dolares = evaluacion_ridge_poly * -1
print(evaluacion_ridge_poly_dolares)

[53022.55185284 50193.47763852 50227.66850762 55985.82108193
 50350.07115558 52941.91056966 54494.29670029 54224.338192
 50726.6136979  53073.83112584]


In [54]:
rmse_promedio_ridge_poly = evaluacion_ridge_poly_dolares.mean()
rmse_std_ridge_poly = evaluacion_ridge_poly_dolares.std()
print(rmse_promedio_ridge_poly)
print(rmse_std_ridge_poly)

52524.0580522172
1953.048351293838


In [55]:
media_precio_ridge_poly = y_train_filtrado.mean()
mediana_precio_ridge_poly = y_train_filtrado.median()
print(media_precio_ridge_poly)
print(mediana_precio_ridge_poly)

189466.25763191588
170800.0


In [ ]:
# Practicamente igual a Polynomial sin Ridge: con grado 2, Ridge aun no tiene mucho que corregir
porcentaje_promedio_ridge_poly = rmse_promedio_ridge_poly / mediana_precio_ridge_poly * 100
print(porcentaje_promedio_ridge_poly)

30.751790428698595


**Resultado:** RMSE promedio de 30.75% — prácticamente igual al de `PolynomialFeatures` solo (30.75%). Con `alpha=1.0`, la regularización de Ridge no cambia casi nada el resultado frente a Polynomial sin regularizar: con grado 2, el modelo aún no tiene pesos lo bastante grandes como para que Ridge tenga mucho que corregir.


---
---

### Probando Lasso

**Qué se va a hacer:** probar `Lasso` (alpha=0.1, sin `PolynomialFeatures` todavía) — a diferencia de Ridge (regularización L2), Lasso usa regularización L1, que puede llevar coeficientes exactamente a cero. Se sube `max_iter` a 10,000 por un posible problema de convergencia con la escala de las columnas transformadas por log.


In [ ]:
# Modelo 5: ColumnTransformer + Lasso (alpha=0.1). max_iter alto por problema de convergencia (ver nota mas abajo)
pipeline_lasso = Pipeline([
    ("transformer", column_transformer),
    ("regresor", Lasso(alpha=0.1, max_iter=10000))
])

In [71]:
evaluacion_lasso = cross_val_score(pipeline_lasso, X_train_filtrado, y_train_filtrado, cv=10, scoring="neg_root_mean_squared_error")

c:\Users\UserGPC\anaconda3\envs\ml_env-3.12\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.315774e+12, tolerance: 1.229e+10
  model = cd_fast.enet_coordinate_descent(
c:\Users\UserGPC\anaconda3\envs\ml_env-3.12\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.179130e+11, tolerance: 1.225e+10
  model = cd_fast.enet_coordinate_descent(
c:\Users\UserGPC\anaconda3\envs\ml_env-3.12\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consi

In [72]:
evaluacion_lasso_dolares = evaluacion_lasso * -1
print(evaluacion_lasso_dolares)

[62028.26219915 58704.16962526 58604.07494082 64244.39925061
 58809.67994037 61934.99510006 62487.13291871 64023.54795319
 60256.23083766 61346.59187743]


In [73]:
rmse_promedio_lasso = evaluacion_lasso_dolares.mean()
rmse_std_lasso = evaluacion_lasso_dolares.std()
print(rmse_promedio_lasso)
print(rmse_std_lasso)

61243.90846432709
1992.442548654183


In [74]:
media_precio_lasso = y_train_filtrado.mean()
mediana_precio_lasso = y_train_filtrado.median()
print(media_precio_lasso)
print(mediana_precio_lasso)

189466.25763191588
170800.0


In [ ]:
# Con alpha=0.1 Lasso no converge bien (ConvergenceWarning) por la escala residual de las columnas log; resultado no confiable, se retoma mas abajo con GridSearchCV
porcentaje_promedio_lasso = rmse_promedio_lasso / mediana_precio_lasso * 100
print(porcentaje_promedio_lasso)

35.85708926482851


**Resultado:** RMSE promedio de 35.86% — prácticamente idéntico a `LinearRegression` y a `Ridge`. El warning de convergencia (`ConvergenceWarning`) con `alpha=0.1` indica que el optimizador no logró estabilizarse del todo con la escala actual de las columnas log — este resultado no es del todo confiable y se retoma más abajo con `GridSearchCV`, buscando un alpha que converja limpio.


---
---

### Probando ElasticNet

**Qué se va a hacer:** probar `ElasticNet` (alpha=0.1, l1_ratio=0.5, sin `PolynomialFeatures` todavía) — combina las penalizaciones de Ridge (L2) y Lasso (L1) en un solo modelo, para comparar contra los resultados individuales de Ridge y Lasso.


In [ ]:
# Modelo 6: ColumnTransformer + Elastic Net (mezcla 50/50 entre Ridge y Lasso)
pipeline_elastic = Pipeline([
    ("transformer", column_transformer),
    ("regresor", ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000, random_state=42))
])

In [79]:
evaluacion_elastic = cross_val_score(pipeline_elastic, X_train_filtrado, y_train_filtrado, cv=10, scoring="neg_root_mean_squared_error")

In [80]:
evaluacion_elastic_dolares = evaluacion_elastic * -1
print(evaluacion_elastic_dolares)

[62885.13633781 59669.78056104 59608.66024233 65080.19987614
 59319.14368942 63048.97460468 63451.54599388 64635.3657846
 61763.88911779 62385.70466079]


In [81]:
rmse_promedio_elastic = evaluacion_elastic_dolares.mean()
rmse_std_elastic = evaluacion_elastic_dolares.std()
print(rmse_promedio_elastic)
print(rmse_std_elastic)

62184.84008684894
1965.244534684047


In [82]:
media_precio_elastic = y_train_filtrado.mean()
mediana_precio_elastic = y_train_filtrado.median()
print(media_precio_elastic)
print(mediana_precio_elastic)

189466.25763191588
170800.0


In [ ]:
# Elastic Net empeora respecto a la linea base: penaliza un modelo que no tenia multicolinealidad real que corregir
porcentaje_promedio_elastic = rmse_promedio_elastic / mediana_precio_elastic * 100
print(porcentaje_promedio_elastic)

36.40798599932607


**Resultado:** RMSE promedio de 36.41% — el peor resultado de los seis modelos probados hasta ahora, incluso peor que la línea base de `LinearRegression` (35.86%). ElasticNet combina las penalizaciones de Ridge y Lasso, pero sin `PolynomialFeatures` no hay sobreajuste ni multicolinealidad real que corregir — la regularización solo quita capacidad al modelo sin ganar nada a cambio.


---
---

### Probando GridSearchCV sobre el mejor puntaje que es PolynomialFeatures 

**Qué se va a hacer:** en vez de seguir fijando `degree=2` y `alpha=1.0` a mano, se va a usar `GridSearchCV` para probar varias combinaciones de `degree` (1, 2, 3) y `alpha` (0.01 a 100) sobre el pipeline Ridge+Polynomial, y quedarse con la combinación que dé el menor RMSE promedio en CV.


In [ ]:
# GridSearchCV sobre Polynomial+Ridge: busca el mejor degree (1,2,3) y alpha (0.01 a 100) combinados
param_grid = [
    {"poly__degree": [1, 2, 3],
    "ridge__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
]

buscando_mejor  = GridSearchCV(pipeline_ridge_poly, param_grid, cv=10, scoring='neg_root_mean_squared_error')

In [ ]:
# Entrena las 15 combinaciones x 10 particiones (150 entrenamientos en total)
buscando_mejor.fit(X_train_filtrado, y_train_filtrado)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...e', Ridge())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'poly__degree': [1, 2, ...], 'ridge__alpha': [0.01, 0.1, ...]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",10
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Con

In [ ]:
# Mejor combinacion encontrada: degree=3, alpha=100 (necesita mucha regularizacion porque grado 3 da muchisima flexibilidad)
print(buscando_mejor.best_params_)
print(buscando_mejor.best_score_)

{'poly__degree': 3, 'ridge__alpha': 100.0}
-50970.531245782455


In [88]:
score_dolares = -buscando_mejor.best_score_
print(score_dolares)

50970.531245782455


In [ ]:
# Mejor resultado de todos hasta ahora: 29.8%   
porcentaje_grid = score_dolares / mediana_precio * 100
print(porcentaje_grid)

29.84223140853774


**Resultado:** la mejor combinación fue degree=3, alpha=100 — **29.84%**, el mejor resultado de toda la ronda hasta este punto. Que alpha=100 (regularización fuerte) ganara junto con degree=3 (el grado más flexible) tiene sentido: a mayor flexibilidad del polinomio, más necesidad de controlar el sobreajuste con una penalización fuerte.


---
---

### Probando GridSearchCV buscando la mejor combinacion de alpha y verificar cual de todos los probados es mejor

**Qué se va a hacer:** antes de descartar Lasso por completo, se va a usar `GridSearchCV` para buscar un `alpha` que logre converger sin el problema de escala visto arriba (`ConvergenceWarning` con alpha=0.1) — para confirmar si el mal resultado de Lasso fue por el valor de alpha o porque el modelo lineal simplemente no mejora con regularización sola.


In [ ]:
# GridSearchCV sobre Lasso: busca el alpha que logre converger sin el problema de escala visto antes
param_grid = [
    {"regresor__alpha": [0.001, 0.01, 0.1, 1.0, 10.0]}
]

buscar_mejor = GridSearchCV(pipeline_lasso, param_grid, cv=10, scoring="neg_root_mean_squared_error")

In [91]:
buscar_mejor.fit(X_train_filtrado, y_train_filtrado)

c:\Users\UserGPC\anaconda3\envs\ml_env-3.12\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.315618e+12, tolerance: 1.229e+10
  model = cd_fast.enet_coordinate_descent(
c:\Users\UserGPC\anaconda3\envs\ml_env-3.12\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.979603e+12, tolerance: 1.225e+10
  model = cd_fast.enet_coordinate_descent(
c:\Users\UserGPC\anaconda3\envs\ml_env-3.12\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consi

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...iter=10000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'regresor__alpha': [0.001, 0.01, ...]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",10
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of inf

In [ ]:
# alpha=1.0 es el que logra converger limpio (sin warning)
print(buscar_mejor.best_params_)
print(buscar_mejor.best_score_)

{'regresor__alpha': 1.0}
-61243.824180681455


In [93]:
score_dolares = -buscar_mejor.best_score_
print(score_dolares)

61243.824180681455


In [ ]:
# Aun convergiendo bien, Lasso no supera la linea base: mismo resultado que Regresion Lineal simple (35.9%)
porcentaje_grid = score_dolares / mediana_precio * 100
print(porcentaje_grid)

35.85703991843177


## Evaluación final: modelo ganador contra el test set

**Qué se va a hacer:** de todos los modelos y búsquedas probados, el ganador fue `GridSearchCV` sobre Polynomial+Ridge (degree=3, alpha=100, 29.8% en CV). Se va a tomar ese `best_estimator_` — ya reentrenado por `GridSearchCV` con el 100% de `X_train_filtrado` — y evaluarlo **una sola vez** contra `X_test_filtrado`/`y_test_filtrado`, el test set nunca antes tocado, para obtener una medida honesta de generalización.


**Resultado:** con `alpha=1.0`, Lasso converge limpio (sin `ConvergenceWarning`) pero el RMSE (35.86%) es prácticamente idéntico al de `LinearRegression` simple — confirma que Lasso, igual que Ridge y ElasticNet, no ayuda nada si el modelo sigue siendo lineal sin `PolynomialFeatures`. El ganador de toda la ronda sigue siendo `GridSearchCV` sobre Polynomial+Ridge, con **29.8%**.


---
---

In [ ]:
# Modelo ganador: Polynomial grado 3 + Ridge alpha=100, ya reentrenado por GridSearchCV con el 80% completo de train
modelo_final = buscando_mejor.best_estimator_

In [ ]:
# Primera y unica vez que se usa X_test_filtrado: prediccion sobre datos nunca vistos
y_pred = modelo_final.predict(X_test_filtrado)

In [ ]:
# Verificar que cada prediccion tiene su precio real correspondiente
len(y_pred) == len(y_test_filtrado)

True

In [ ]:
# RMSE final, comparando prediccion contra el precio real de test (y_true primero, y_pred despues)
rmse_test = root_mean_squared_error(y_test_filtrado, y_pred)

In [102]:
print(rmse_test)

50322.55641235826


In [ ]:
# Resultado final: 29.5% en test, muy cerca del 29.8% visto en cross-validation - confirma que el modelo generaliza bien, sin overfitting
porcentaje_promedio_rmse = rmse_test / mediana_precio * 100
print(porcentaje_promedio_rmse)

29.46285504236432


In [16]:
X_train_filtrado.to_csv("../data/raw/X_train_filtrado.csv", index=False)
y_train_filtrado.to_csv("../data/raw/y_train_filtrado.csv", index=False)

## Cierre: dataset filtrado guardado para la Ronda 2

**Qué se hizo:** se guardaron `X_train_filtrado`/`y_train_filtrado` (14,839 filas, ya sin los outliers estructurales de precio tope y edad tope) en `data/raw/`, como punto de partida para la Ronda 2 (`03_diagnostico_y_mejora.ipynb`). La Ronda 2 vuelve a partir este mismo conjunto en su propio train/test (con un `random_state` distinto), así que el test set de esta Ronda 1 (`X_test_filtrado`/`y_test_filtrado`, usado una sola vez arriba) no se reutiliza — queda consumido aquí.
